In [1]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)



In [2]:


def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation

generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)


In [3]:

import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [4]:
import torchaudio
import glob

# 1. Get the path to any of the newly generated .wav files
wav_files = glob.glob('/kaggle/working/synthetic_mashups/train/*/*.wav')
sample_file = wav_files[0]

# 2. Load the audio file
waveform, sample_rate = torchaudio.load(sample_file)

# 3. Print the tensor shape in tuple format
print(tuple(waveform.shape))


(2, 661500)


In [5]:
import glob
import torch

pt_files = glob.glob("/kaggle/working/features/train/*/*.pt")
sample_pt = pt_files[0]

x = torch.load(sample_pt)

print("file:", sample_pt)
print("shape:", tuple(x.shape))
print("dtype:", x.dtype)

file: /kaggle/working/features/train/blues/mashup_024.pt
shape: (2, 128, 1292)
dtype: torch.float32


## CRNN
*With batch_first=True, PyTorch LSTM expects input shaped (batch, seq_len, input_size), so we keep time as the sequence axis before sending data into the LSTM.*

In [6]:
import torch
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes=10, n_mels=128, hidden_size=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        lstm_input_size = 64 * (n_mels // 4)

        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = self.features(x)                  # (B, 64, F', T')
        b, c, f, t = x.shape

        x = x.permute(0, 3, 1, 2)            # (B, T', C, F')
        x = x.reshape(b, t, c * f)           # (B, T', C*F)

        x, _ = self.lstm(x)                  # (B, T', 2*hidden_size)
        x = torch.max(x, dim=1).values       # global max pool over time
        x = self.fc(x)
        return x


In [7]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# load one real saved feature tensor
x = torch.load(sample_pt).float()   # from Q3

# make sure it is 3D: (C, MELS, TIME)
print("single feature shape:", tuple(x.shape))

# create a fake batch of 32 using the same shape
batch = x.unsqueeze(0).repeat(32, 1, 1, 1)   # (32, C, MELS, TIME)

cnn_backbone = nn.Sequential(
    nn.Conv2d(2, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

with torch.no_grad():
    out = cnn_backbone(batch)

print("Q4 shape:", tuple(out.shape))


single feature shape: (2, 128, 1292)
Q4 shape: (32, 64, 32, 323)


In [8]:
model = CRNN(num_classes=10, n_mels=x.shape[1], hidden_size=64)

lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print("Q5 LSTM params:", lstm_params)


Q5 LSTM params: 1082368


In [9]:
import torch
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)                 # (B, 64, 32, 323)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)      # (B, 323, 64, 32)
        x = x.reshape(b, t, c * f)     # (B, 323, 2048)
        x, _ = self.lstm(x)
        x = torch.max(x, dim=1).values
        x = self.fc(x)
        return x

model = CRNN()
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

print("Q5 answer:", lstm_params)


Q5 answer: 1082368


In [10]:
for name, p in model.lstm.named_parameters():
    print(name, tuple(p.shape), p.numel())

weight_ih_l0 (256, 2048) 524288
weight_hh_l0 (256, 64) 16384
bias_ih_l0 (256,) 256
bias_hh_l0 (256,) 256
weight_ih_l0_reverse (256, 2048) 524288
weight_hh_l0_reverse (256, 64) 16384
bias_ih_l0_reverse (256,) 256
bias_hh_l0_reverse (256,) 256
